In [1]:
from classy import Class

import os
import copy
import yaml
import matplotlib.pyplot as plt
import numpy as np
import matplotlib

from matplotlib import rc
from scipy.interpolate import interp1d

rc('font',**{'family':'serif','serif':['Times']})
rc('text', usetex=True)
#matplotlib.rc('font', **font)
matplotlib.rcParams['legend.fontsize']='medium'
plt.rcParams["figure.figsize"] = [8.0,6.0]

# warmup

run a fiducial cosmology

In [2]:
params_EDE = {

    "output": "tCl,lCl,mPk",
    # LCDM parameters
    "omega_b": 0.02251,
    "omega_cdm": 0.1320,
    "H0": 72.81,
    "tau_reio": 0.068,
    "A_s": 2.191e-9,
    "n_s": 0.9860,

    # neutrinos
    "N_ur": 2.0328,
    "N_ncdm": 1,
    "deg_ncdm": 1,
    "m_ncdm": 0.06,
    "T_ncdm": 0.71611,

    # EDE parameters
    "scf_potential": "axion",
    "n_axion": 2.6,
    "f_axion": 0.1,
    "m_axion": 1e4,
    "scf_parameters": '2.72,0',

    # extra EDE parameters
    "scf_evolve_as_fluid": "no",
    "scf_evolve_like_axionCAMB": "no",
    "do_shooting": "no",
    "do_shooting_scf": "no",
    "attractor_ic_scf": "no",

    # verbosity
    "input_verbose": 1,
    "background_verbose": 1,
    "thermodynamics_verbose": 1,
    "perturbations_verbose": 1,
    "transfer_verbose": 1,
    "primordial_verbose": 1,
    #"spectra_verbose": 1,
    # "nonlinear_verbose": 1,
    "lensing_verbose": 1,
    "output_verbose": 1,
    "l_max_scalars":5000,
}


In [3]:
params_mEDE = {
    # "compute_sigma8": "yes",

    "output": "tCl,lCl,mPk",
    "omega_b": 0.02251,
    "omega_cdm": 0.1320,
    "H0": 72.81,
    "tau_reio": 0.068,
    "A_s": 2.191e-9,
    "n_s": 0.9860,

    "N_ur": 2.0328,
    "N_ncdm": 1,
    "deg_ncdm": 1,
    "m_ncdm": 0.06,
    "T_ncdm": 0.7161,

    "N_mscf": 6,
    "n_axion_mscf": '2.6, 2.6, 2.6, 2.6, 2.6, 2.6',
    "f_axion_mscf": '0.1, 0.1, 0.1, 0.1, 0.1, 0.1',
    "m_mscf": '1e3, 5e3, 1e4, 5e4, 1e5, 5e5',
    "theta_ini_mscf": '2.72, 2.72, 2.72, 2.72, 2.72, 2.72',
    "theta_prime_ini_mscf": '0.0, 0.0, 0.0, 0.0, 0.0, 0.0',

    "input_verbose": 1,
    "background_verbose": 1,
    "thermodynamics_verbose": 1,
    "perturbations_verbose": 1,
    "transfer_verbose": 1,
    "primordial_verbose": 1,
    # "spectra_verbose": 1,
    # "nonlinear_verbose": 1,
    "lensing_verbose": 1,
    "output_verbose": 1,
    "l_max_scalars":5000,
}


In [ ]:
EDE1 = Class()
EDEm = Class()
EDE1.set(params_EDE)
EDEm.set(params_mEDE)

In [ ]:
#LCDM.compute()
EDE1.compute()
EDEm.compute()

#ADE1.get_background()
#ADE2.get_background()

In [ ]:
bg1 = EDE1.get_background()
print(bg1.keys())
bg_m = EDEm.get_background()
print(bg_m.keys())
derived = EDE1.get_current_derived_parameters(['z_eq','z_rec'])

In [ ]:
# Prepare the data
z = bg_m['z']

fig, ax = plt.subplots()
for i in range(params_mEDE['N_mscf']):  # Corrected to use params_mEDE
    rho_EDE_m = bg_m[f'(.)rho_mscf[{i}]']
    f_EDE_m = rho_EDE_m / bg_m['(.)rho_tot']
    ax.plot(1+z, f_EDE_m, label=f'$f_{{EDE}}$ (i={i})')
    
ax.set_xlabel(r'$1+z$',fontsize='large')
ax.set_ylabel(r'$f_{EDE}$',fontsize='large')
#ax.set_ylim(0, .1)  # Set based on physical expectations
#ax.tick_params(colors='darkred')

lines, labels = ax.get_legend_handles_labels()
ax.legend(lines, labels, loc='upper right', bbox_to_anchor=(1, 1))
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(1e0,1e5)
ax.set_ylim(1e-7,1e-1)
plt.title("mAxiCLASS: Fraction of EDE in each field (theory parameters)")
plt.tight_layout()
plt.show()

# Compute $C_l$ 

In [ ]:
fig, ax = plt.subplots()
plt.xlim([2,3000])
plt.xlabel(r"$\ell$")
plt.ylabel(r"$\ell (\ell+1) C_l^{TT} / 2 \pi \,\,\, [\times 10^{10}]$")
plt.grid()
pi = np.pi#

cl_tot = EDEm.raw_cl(3000)
cl_tot_1 = EDE1.raw_cl(3000)
# cl_lensed = EDEm.lensed_cl(3000)
ell = cl_tot['ell']
ell_1 = cl_tot_1['ell']
factor = 1.e10*ell*(ell+1.)/2./pi
plt.semilogx(ell,factor*cl_tot['tt'],'r-',label=r'$C_l^{TT}$ (MSCF)')
plt.semilogx(ell,factor*cl_tot_1['tt'],'b-',label=r'$C_l^{TT}$ (1SCF)')

plt.legend()
    

# Matter power spectrum

In [ ]:
kk = np.logspace(-4, 0, 10000)
Pk_axion1 = [EDE1.pk(ki, 0) for ki in kk]

Pk_axion_m = [EDEm.pk(ki, 0) for ki in kk]

fig, ax = plt.subplots()
plt.plot(kk,Pk_axion1,label=r'$P_k$ (MSCF)')
plt.plot(kk,Pk_axion_m,label=r'$P_k$ (1SCF)')
ax.set_xscale('log')
ax.set_yscale('log')
plt.legend()

In [26]:
EDE1.empty()
EDEm.empty()